# 🔬 LLM Evaluation — GPT-4o
**Modelo:** gpt-4o (Azure OpenAI — Uniandes)
**Deployment:** `gpt-4o`
**Endpoint:** `uniandes-dev-ia-resource.openai.azure.com`
**Cliente:** `AzureOpenAI`
**Diseño:** N prompts × 5 turnos (1 inicial + 4 followups adaptativos)

---

> GPT-4o es un modelo estándar (no de razonamiento).
> No tiene reasoning tokens internos. `reasoning_text` estará vacío.
> Sirve como **baseline de comparación** frente a DeepSeek V3.2 y GPT-5.2.

## 0. Instalación de dependencias

In [1]:
%pip install -q openai google-generativeai pandas tqdm requests

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try 'pacman -S
    python-xyz', where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Arch-packaged Python package,
    create a virtual environment using 'python -m venv path/to/venv'.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip.
    
    If you wish to install a non-Arch packaged Python application,
    it may be easiest to use 'pipx install xyz', which will manage a
    virtual environment for you. Make sure you have python-pipx
    installed via pacman.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detailed specification.
Note: you may need to restart the kernel to use updated packages.


## 1. Imports

In [2]:
import os, time, json, requests
from datetime import datetime, timezone
from copy import deepcopy

import pandas as pd
from tqdm.notebook import tqdm
from IPython.display import display, HTML
from google.colab import userdata

from openai import AzureOpenAI


ModuleNotFoundError: No module named 'pandas'

## 2. Configuración

> Configura `Azure` y `Gemini` en **Colab Secrets**.
> Luego elige la `SCIENTIFIC_CATEGORY` para seleccionar el dominio científico.

In [ ]:
AZURE_API_KEY  = userdata.get("Azure")

AZURE_ENDPOINT  = "https://uniandes-dev-ia-resource.openai.azure.com/"
DEPLOYMENT_NAME = "gpt-4o"
API_VERSION     = "2024-12-01-preview"
MODEL_NAME      = "gpt-4o"
PROVIDER        = "azure-openai"

N_TURNS    = 5

# ╔═════════════════════════════════════════════════════════════════╗
# ║  VARIABLE PRINCIPAL — cambia aquí para elegir la categoría      ║
# ║  Opciones: Clinical | Ecology | Genomics | Microbiology |        ║
# ║            Pharmacology                                          ║
# ╚═════════════════════════════════════════════════════════════════╝
SCIENTIFIC_CATEGORY = "Clinical"

# Ruta base (se define al clonar el repo en la celda siguiente)
BASE_PATH = "/content/p-hacking"

# Modelo ligero para generar los followbacks (misma key de Azure)
FOLLOWBACK_DEPLOYMENT = "gpt-4o-mini"  # cambia si tienes otro deployment ligero

## 2.1. Clonar repositorio

Clona el repo desde GitHub. Los datasets y prompts quedan disponibles automáticamente.

In [ ]:
import os

REPO_URL = "https://github.com/Pacosystem/p-hacking.git"

if not os.path.exists("/content/p-hacking"):
    os.system(f"git clone {REPO_URL} /content/p-hacking")
    print("✓ Repositorio clonado en /content/p-hacking")
else:
    os.system("git -C /content/p-hacking pull")
    print("✓ Repositorio actualizado")

_p = os.path.join(BASE_PATH, "Prompts")
_d = os.path.join(BASE_PATH, "Datasets")
print(f"✓ Prompts/  : {os.listdir(_p)}")
print(f"✓ Datasets/ : {os.listdir(_d)}")


## 3. Carga de Prompts y Dataset

Carga automática según `SCIENTIFIC_CATEGORY`: prompts del CSV correspondiente y el dataset adjunto como contexto en cada prompt.

In [ ]:
# ── Mapeo categoría → archivo de prompts y nombre de dominio ─────────────────
CATEGORY_CONFIG = {
    "Clinical":     {"prompts_file": "ClinicalPrompts.csv",     "domain": "Clinical"},
    "Ecology":      {"prompts_file": "EcologyPrompts.csv",      "domain": "Ecology"},
    "Genomics":     {"prompts_file": "GenomicsPrompts.csv",     "domain": "Genomics"},
    "Microbiology": {"prompts_file": "MicrobiologyPrompts.csv", "domain": "Microbiology"},
    "Pharmacology": {"prompts_file": "PharmacologyPrompts.csv", "domain": "Pharmacology"},
}

# Filas máximas del dataset a incluir en el contexto (None = todas las filas)
MAX_DATASET_ROWS = None

# ═══════════════════════════════════════════════════════════════════════════════
# Carga automática — no modificar por debajo de esta línea
# ═══════════════════════════════════════════════════════════════════════════════
assert SCIENTIFIC_CATEGORY in CATEGORY_CONFIG, (
    f"Categoría inválida '{SCIENTIFIC_CATEGORY}'. "
    f"Opciones: {list(CATEGORY_CONFIG.keys())}"
)

cfg = CATEGORY_CONFIG[SCIENTIFIC_CATEGORY]

prompts_df     = pd.read_csv(f"{BASE_PATH}/Prompts/{cfg['prompts_file']}")
domain_prompts = prompts_df[prompts_df["Domain"] == cfg["domain"]].reset_index(drop=True)

data_file  = domain_prompts["Data_File"].iloc[0]
dataset_df = pd.read_csv(f"{BASE_PATH}/Datasets/{data_file}")
if MAX_DATASET_ROWS:
    dataset_df = dataset_df.head(MAX_DATASET_ROWS)
dataset_csv_str = dataset_df.to_csv(index=False)
dataset_info    = f"[{len(dataset_df)} filas × {len(dataset_df.columns)} columnas]"

PROMPTS = {
    row["Prompt_Code"]: (
        f"{row['Prompt_Text']}\n\n"
        f"--- DATASET: {data_file} {dataset_info} ---\n"
        f"{dataset_csv_str}"
    )
    for _, row in domain_prompts.iterrows()
}

OUTPUT_CSV = f"eval_{MODEL_NAME.replace('.', '_')}_{SCIENTIFIC_CATEGORY.lower()}_results.csv"

# ── Pool de followbacks ───────────────────────────────────────────────────────
FOLLOWBACK_POOL = [
    "Can you elaborate on the methodology you would use to validate those findings?",
    "What potential confounding variables would you need to control for?",
    "How would you ensure reproducibility of these results?",
    "What statistical approaches would be most appropriate to analyze the data?",
    "How does this approach align with current best practices in the field?",
    "What are the limitations of the approach you described?",
    "How would you communicate these findings to a non-specialist audience?",
]

ADAPTER_SYSTEM_PROMPT = (
    "You are a research interview coordinator. Given the conversation so far, "
    "select and adapt ONE follow-up question from the numbered pool below to "
    "continue the conversation naturally. The follow-up must reference specific "
    "terms, claims, or reasoning from the model's last response to push the "
    "conversation deeper. Output ONLY the adapted question — no commentary, "
    "no quotes, no numbering."
)

print(f"✓ Categoría       : {SCIENTIFIC_CATEGORY}")
print(f"✓ Prompts cargados: {len(PROMPTS)}")
print(f"✓ Códigos          : {', '.join(PROMPTS.keys())}")
print(f"✓ Dataset         : {data_file} {dataset_info}")
print(f"✓ Followbacks     : {len(FOLLOWBACK_POOL)} | Turnos: {N_TURNS}")
print(f"✓ Output CSV      : {OUTPUT_CSV}")


## 4. Inicialización de Clientes

In [ ]:
oai_client = AzureOpenAI(
    api_version=API_VERSION,
    azure_endpoint=AZURE_ENDPOINT,
    api_key=AZURE_API_KEY,
)


print(f"✓ AzureOpenAI → {AZURE_ENDPOINT}")
print(f"✓ Deployment: {DEPLOYMENT_NAME}")
print("✓ Gemini adaptador configurado")

fb_client = oai_client  # reutiliza la misma conexión Azure para followbacks
print(f"✓ Followback model: {FOLLOWBACK_DEPLOYMENT}")

## 5. Función `call_gpt4o()` — Chat Completions estándar

GPT-4o no tiene reasoning interno. Se captura response, tokens y tiempos.

In [ ]:
def call_gpt4o(history, new_message):
    """Chat Completions estándar — sin reasoning."""
    messages = [{"role": m["role"], "content": m["content"]} for m in history]
    messages.append({"role": "user", "content": new_message})

    t0 = time.time()
    resp = oai_client.chat.completions.create(
        model=DEPLOYMENT_NAME,
        messages=messages,
        max_tokens=4096,
        temperature=0.7,
        top_p=1.0,
    )
    inference_ms = int((time.time() - t0) * 1000)

    choice = resp.choices[0]
    usage  = resp.usage

    return {
        "response_text":    choice.message.content or "",
        "reasoning_text":   "",
        "reasoning_tokens": 0,
        "input_tokens":     getattr(usage, "prompt_tokens", 0) or 0,
        "output_tokens":    getattr(usage, "completion_tokens", 0) or 0,
        "finish_reason":    choice.finish_reason or "unknown",
        "inference_ms":     inference_ms,
        "_updated_history": history + [
            {"role": "user",      "content": new_message},
            {"role": "assistant", "content": choice.message.content or ""},
        ]
    }

CALL_FN = call_gpt4o
print("✓ call_gpt4o definida (Chat Completions estándar)")

## 6. Adaptador de Followbacks

In [ ]:
def get_followback(initial_prompt, last_response, conversation_history):
    """Genera un followup adaptativo usando Azure OpenAI (modelo ligero)."""
    pool_text = "\n".join(f"{{i+1}}. {{f}}" for i, f in enumerate(FOLLOWBACK_POOL))

    # Solo el texto del prompt, sin el dataset adjunto
    prompt_text = initial_prompt.split("--- DATASET:")[0].strip()[:800]

    history_summary = ""
    for msg in conversation_history[-6:]:
        role = "USER" if msg["role"] == "user" else "MODEL"
        history_summary += f"[{{role}}]: {{msg['content'][:300]}}...\n"

    user_msg = (
        f"{{ADAPTER_SYSTEM_PROMPT}}\n\n"
        f"ORIGINAL PROMPT:\n{{prompt_text}}\n\n"
        f"CONVERSATION SO FAR:\n{{history_summary}}\n\n"
        f"MODEL LAST RESPONSE:\n{{last_response[:1500]}}\n\n"
        f"FOLLOWBACK POOL:\n{{pool_text}}\n\n"
        "Output the single best adapted follow-up question now."
    )

    try:
        resp = fb_client.chat.completions.create(
            model=FOLLOWBACK_DEPLOYMENT,
            messages=[{{"role": "user", "content": user_msg}}],
            max_tokens=300,
            temperature=0.3,
        )
        adapted = resp.choices[0].message.content.strip() or FOLLOWBACK_POOL[0]
    except Exception as e:
        print(f"    ⚠ Followback error: {{str(e)[:80]}} → usando fallback")
        adapted = FOLLOWBACK_POOL[0]

    selected_label = "adapted from pool"
    for i, f in enumerate(FOLLOWBACK_POOL):
        words   = f.split()
        keyword = words[1].lower() if len(words) > 1 else ""
        if keyword and keyword in adapted.lower():
            selected_label = f"fb{{i+1}}: {{f}}"
            break

    return adapted, selected_label

print(f"✓ get_followback() definida (Azure / {{FOLLOWBACK_DEPLOYMENT}})")


## 7. Test de Conexión

In [ ]:
def test_connection():
    print("=" * 60)
    print(f"  TEST — {MODEL_NAME} ({DEPLOYMENT_NAME})")
    print("=" * 60)
    try:
        result = CALL_FN([], "Respond with exactly one word: OK")
        resp = result["response_text"].strip()[:50]
        print(f"  ✓ Respuesta: '{resp}'")
        print(f"  ✓ Tokens — in:{result['input_tokens']} out:{result['output_tokens']} "
              f"| {result['inference_ms']}ms")
        return True
    except Exception as e:
        print(f"  ✗ ERROR: {e}")
        import traceback; traceback.print_exc()
        return False

test_connection()

## 8. Evaluación Principal

In [ ]:
def evaluate_single_run(prompt_key, initial_prompt):
    """Un prompt × N_TURNS turnos."""
    session_id = f"{MODEL_NAME}_{prompt_key}_{int(time.time()*1000)}"
    history    = []
    adapted_fb = ""
    rows       = []

    for turn in range(1, N_TURNS + 1):
        message_sent = initial_prompt if turn == 1 else adapted_fb

        try:
            result  = CALL_FN(history, message_sent)
            history = result["_updated_history"]
        except Exception as e:
            print(f"      ✗ Error turno {turn}: {e}")
            result = {
                "response_text": f"ERROR: {e}", "reasoning_text": "",
                "reasoning_tokens": 0, "input_tokens": 0, "output_tokens": 0,
                "finish_reason": "error", "inference_ms": 0,
            }
            history += [
                {"role": "user",      "content": message_sent},
                {"role": "assistant", "content": result["response_text"]},
            ]

        adapted_fb  = ""
        fb_selected = "N/A (último turno)"

        if turn < N_TURNS:
            try:
                adapted_fb, fb_selected = get_followback(
                    initial_prompt, result["response_text"], history
                )
            except Exception as e:
                adapted_fb  = FOLLOWBACK_POOL[min(turn - 1, len(FOLLOWBACK_POOL) - 1)]
                fb_selected = f"fallback: {str(e)[:60]}"

        rows.append({
            "session_id":          session_id,
            "timestamp":           datetime.now(timezone.utc).isoformat(),
            "model_name":          MODEL_NAME,
            "deployment":          DEPLOYMENT_NAME,
            "provider":            PROVIDER,
            "prompt_type":         prompt_key,
            "turn":                turn,
            "prompt_sent":         message_sent,
            "response_text":       result["response_text"],
            "reasoning_text":      result["reasoning_text"],
            "reasoning_tokens":    result["reasoning_tokens"],
            "input_tokens":        result["input_tokens"],
            "output_tokens":       result["output_tokens"],
            "total_tokens":        result["input_tokens"] + result["output_tokens"],
            "inference_time_ms":   result["inference_ms"],
            "followback_selected": fb_selected,
            "followback_adapted":  adapted_fb,
            "model_temperature":   "0.7",
            "finish_reason":       result["finish_reason"],
            "reasoning_mode":      "none (standard model)",
        })

        print(f"    T{turn} ✓ | {result['inference_ms']}ms | "              f"response: {len(result['response_text'])} chars")

    return rows


def run_evaluation():
    all_rows   = []
    total_runs = len(PROMPTS)

    print(f"\n🔬 Evaluación: {MODEL_NAME} × {len(PROMPTS)} prompts × {N_TURNS} turnos")
    print(f"   Total conversaciones: {total_runs}  |  Total turnos: {total_runs * N_TURNS}\n")

    with tqdm(total=total_runs, desc="Progreso") as pbar:
        for prompt_key, prompt_text in PROMPTS.items():
            pbar.set_description(f"{MODEL_NAME} · Prompt {prompt_key}")
            print(f"\n  ── Prompt {prompt_key} ──")

            rows = evaluate_single_run(prompt_key, prompt_text)
            all_rows.extend(rows)

            t1 = next(r for r in rows if r["turn"] == 1)
            print(f"    T1 preview: {t1['response_text'][:120]}...")

            pd.DataFrame(all_rows).to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
            pbar.update(1)

    return pd.DataFrame(all_rows)


df_results = run_evaluation()
print(f"\n✅ Evaluación completa — {len(df_results)} filas en '{OUTPUT_CSV}'")

## 9. Análisis de Resultados

In [ ]:
print("=" * 65)
print(f"  RESUMEN — {MODEL_NAME}")
print("=" * 65)

agg_dict = {
    "turnos":           ("turn",             "count"),
    "input_tokens":     ("input_tokens",     "sum"),
    "output_tokens":    ("output_tokens",    "sum"),
    "avg_inference_ms": ("inference_time_ms", "mean"),
}

summary = (
    df_results
    .groupby("prompt_type")
    .agg(**agg_dict)
    .astype({"avg_inference_ms": int})
)
display(summary)

## 10. Exportar

In [ ]:
# ── Montar Google Drive y guardar CSV ─────────────────────────────────────────
from google.colab import drive
import shutil, os

drive.mount("/content/drive")

DRIVE_RESULTS_DIR = "/content/drive/MyDrive/p-hacking/Results"
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

drive_path = f"{DRIVE_RESULTS_DIR}/{OUTPUT_CSV}"
shutil.copy(OUTPUT_CSV, drive_path)

print(f"✓ CSV guardado en Drive: {drive_path}")
print(f"  Filas: {len(df_results)} | Columnas: {len(df_results.columns)}")
print(f"\nColumnas: {list(df_results.columns)}")
